# Step 9 — 2×2 ablation with nested cross-conformal (CV+) calibration

**RESS 2025 — GAN-Conformal-RUL**

Why: with split conformal, each fold calibrated on only 2 bearings, and coverage fell to 0.79–0.82
overall at a nominal 0.90, even with the true stage (`results/c3_offline`). CV+ (Barber et al., 2021)
uses all 12 non-test bearings for calibration while no model ever scores a bearing it was trained on.

Per outer fold (same 3 test bearings as before): the 12 other bearings form 4 inner groups of 3
(one per operating condition). Inner model *j* trains on 8 bearings (+1 for early stopping), with its
own scaler and its own GAN, and predicts its held-out group (calibration) and the test bearings.

| | real data | + GAN (r = 0.75) |
|---|---|---|
| **single-task** | baseline | +GAN |
| **multi-task (λ = 1.0)** | +MT | +GAN+MT |

Cost: 20 GANs (once, seed-independent), then per seed 5 outer × 4 inner × (1 stage classifier + 4 configs).
Every finished unit is saved; after a disconnect, re-run all cells and it resumes.

* Checkpoints (GANs): `MyDrive/ress_checkpoints/cvplus_gans/`
* Results: `MyDrive/ress_results/09_cvplus/` (`predictions/`, `stage_classifier/`, summaries)

## 1. Setup

In [ ]:
import os, sys, shutil, time, json
os.chdir('/content')
REPO_PATH = '/content/RESS_2025_GAN_Conformal_RUL'
if os.path.exists(REPO_PATH):
    shutil.rmtree(REPO_PATH)
!git clone https://github.com/f-khadija-benzine/RESS_2025_GAN_Conformal_RUL.git {REPO_PATH}
os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH); sys.path.insert(0, f'{REPO_PATH}/src')
from google.colab import drive
drive.mount('/content/drive')

GAN_DIR     = '/content/drive/MyDrive/ress_checkpoints/cvplus_gans'   # model checkpoints
RESULTS_DIR = '/content/drive/MyDrive/ress_results/09_cvplus'         # everything this notebook writes
PRED_DIR    = f'{RESULTS_DIR}/predictions'
STAGE_DIR   = f'{RESULTS_DIR}/stage_classifier'
for p in (GAN_DIR, RESULTS_DIR, PRED_DIR, STAGE_DIR):
    os.makedirs(p, exist_ok=True)

import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
import numpy as np
from data_loader import XJTUSYLoader
from health_indicator_v3 import HealthIndicatorPipeline
from windowing import build_folds
from model import ModelConfig, RULTrainer, evaluate_grid
from multitask import MTConfig, MultiTaskTrainer
from gan import StageGAN, GANConfig
from cvplus import build_inner_folds, prepare_inner, evaluate_outer, K_INNER, ALL_BEARINGS

SEEDS = [1, 2, 3, 4, 5]
LAMBDA = 1.0        # chosen from the multi-task sweep
AUG_RATIO = 0.75    # chosen from the augmentation sweep
ALPHA = 0.10
TARGET = 2          # 0-indexed near-failure stage
CONFIGS = {'baseline': (False, False), '+GAN': (True, False),
           '+MT': (False, True), '+GAN+MT': (True, True)}
TAGS = {'baseline': 'nogan_st', '+GAN': 'gan_st', '+MT': 'nogan_mt', '+GAN+MT': 'gan_mt'}

## 2. Data, outer folds, inner folds

In [ ]:
CANDIDATES = ['/content/drive/MyDrive/XJTU-SY',
              '/content/drive/MyDrive/XJTU-SY_Bearing_Datasets',
              '/content/drive/MyDrive/data/XJTU-SY']
DATA_ROOT = next((p for p in CANDIDATES if os.path.exists(p)), None); assert DATA_ROOT
all_data = XJTUSYLoader(DATA_ROOT).load_all()
results = HealthIndicatorPipeline(fpt_consecutive=5,
            fpt_min_relative_rise=0.20).process_all(all_data, verbose=False)
outer_folds = build_folds(results)

inner_folds = {}
for of in outer_folds:
    k = of['fold']
    inner_folds[k] = build_inner_folds(results, of)
    print(f"Outer fold {k} — test {of['test']}")
    for inf in inner_folds[k]:
        # leakage guards: held-out, validation, training and test are disjoint
        groups = [set(inf['holdout']), set(inf['val']), set(inf['train']), set(of['test'])]
        assert sum(len(g) for g in groups) == len(set().union(*groups)) == 15
        print(f"   inner {inf['inner']}: holdout {inf['holdout']} | val {inf['val']} | {len(inf['train'])} train")

_data_cache = {}
def inner_data(k, j):
    if (k, j) not in _data_cache:
        of = next(f for f in outer_folds if f['fold'] == k)
        _data_cache[(k, j)] = prepare_inner(results, of, inner_folds[k][j - 1])
    return _data_cache[(k, j)]

## 3. One GAN per inner model (trained once, seed-independent, resumable)

In [ ]:
gans = {}
for of in outer_folds:
    k = of['fold']
    for j in range(1, K_INNER + 1):
        path = f'{GAN_DIR}/gan_outer{k}_inner{j}.pt'
        if not os.path.exists(path):
            d = inner_data(k, j)
            t0 = time.time()
            cfg = GANConfig(epochs=300, augment_ratio=1.0, target_stage=3, n_critic=3)
            g = StageGAN(cfg)
            g.fit(d['X_train'], d['y_stage_train'], verbose=False)
            torch.save({'generator': g.G.state_dict(), 'critic': g.D.state_dict(),
                        'config': cfg.__dict__, 'history': g.history}, path)
            print(f'GAN outer {k} inner {j}: trained in {(time.time()-t0)/60:.1f} min', flush=True)
        ck = torch.load(path, map_location='cuda' if torch.cuda.is_available() else 'cpu')
        g = StageGAN(GANConfig(**ck['config']))
        g.G.load_state_dict(ck['generator']); g.D.load_state_dict(ck['critic'])
        gans[(k, j)] = g
print(f'{len(gans)}/20 GANs ready.')

## 4. Helpers — augmentation, shared stage classifier, one inner run

In [ ]:
def augment(gan, d, ratio, seed):
    rng = np.random.default_rng(seed)
    Xtr, ytr, s = d['X_train'], d['y_rul_train'], d['y_stage_train']
    s3 = s == TARGET; n_syn = int(round(ratio * int(s3.sum())))
    if n_syn == 0: return Xtr, ytr, s
    X_syn = gan.sample(n_syn, TARGET)
    y_syn = rng.choice(ytr[s3], size=n_syn, replace=True)
    return (np.concatenate([Xtr, X_syn.astype(np.float32)]),
            np.concatenate([ytr, y_syn.astype(np.float32)]),
            np.concatenate([s, np.full(n_syn, TARGET, dtype=s.dtype)]))


def stage_classifier(k, j, seed):
    # One multi-task network per (outer, inner, seed), real training data only,
    # shared by all four configs. Saved, so a restart reloads it.
    path = f'{STAGE_DIR}/outer{k}_inner{j}_s{seed}.npz'
    if os.path.exists(path):
        z = np.load(path); return z['p_hold'], z['p_test']
    d = inner_data(k, j)
    clf = MultiTaskTrainer(MTConfig(epochs=100, patience=25, lr=5e-4, seed=seed, lambda_stage=LAMBDA))
    clf.fit(d['X_train'], d['y_rul_train'], d['y_stage_train'], d['X_val'], d['y_rul_val'], verbose=False)
    p_hold, p_test = clf.predict_stage_proba(d['X_hold']), clf.predict_stage_proba(d['X_test'])
    np.savez(path, p_hold=p_hold, p_test=p_test)
    return p_hold, p_test


def run_inner(config, k, j, seed):
    # Train one inner model; save its predictions on the held-out group and the test bearings.
    path = f'{PRED_DIR}/{TAGS[config]}_outer{k}_inner{j}_s{seed}.npz'
    if os.path.exists(path):
        return False
    use_gan, use_mt = CONFIGS[config]
    d = inner_data(k, j)
    if use_gan:
        Xtr, ytr, str_ = augment(gans[(k, j)], d, AUG_RATIO, seed)
    else:
        Xtr, ytr, str_ = d['X_train'], d['y_rul_train'], d['y_stage_train']
    extra = {}
    if use_mt:
        tr = MultiTaskTrainer(MTConfig(epochs=100, patience=25, lr=5e-4, seed=seed, lambda_stage=LAMBDA))
        tr.fit(Xtr, ytr, str_, d['X_val'], d['y_rul_val'], verbose=False)
        yp_hold, yp_test = tr.predict_rul(d['X_hold']), tr.predict_rul(d['X_test'])
        extra = {'own_pstage_hold': tr.predict_stage_proba(d['X_hold']),
                 'own_pstage_test': tr.predict_stage_proba(d['X_test'])}
    else:
        tr = RULTrainer(ModelConfig(epochs=100, patience=25, lr=5e-4, seed=seed))
        tr.fit(Xtr, ytr, d['X_val'], d['y_rul_val'], stage_train=str_, stage_val=d['y_stage_val'],
               mask_healthy=False, verbose=False)
        yp_hold, yp_test = tr.predict(d['X_hold']), tr.predict(d['X_test'])
    p_hold, p_test = stage_classifier(k, j, seed)
    np.savez(path, yp_hold=yp_hold, y_hold=d['y_rul_hold'], s_hold=d['y_stage_hold'],
             bid_hold=d['bid_hold'], shat_hold=p_hold.argmax(1),
             yp_test=yp_test, y_test=d['y_rul_test'], s_test=d['y_stage_test'],
             bid_test=d['bid_test'], pstage_test=p_test, **extra)
    return True

## 5. Run everything (resumable)

Order: seed → outer fold → inner model → 4 configs, so every finished seed is complete for all
configs. If Colab disconnects, re-run all cells: finished units are skipped in seconds.

In [ ]:
total = len(SEEDS) * len(outer_folds) * K_INNER * len(CONFIGS)
done = sum(os.path.exists(f'{PRED_DIR}/{TAGS[c]}_outer{f["fold"]}_inner{j}_s{s}.npz')
           for s in SEEDS for f in outer_folds for j in range(1, K_INNER + 1) for c in CONFIGS)
print(f'{done}/{total} inner runs already on Drive')

t_start = time.time()
for seed in SEEDS:
    for of in outer_folds:
        k = of['fold']
        for j in range(1, K_INNER + 1):
            t0 = time.time(); trained = 0
            for config in CONFIGS:
                trained += run_inner(config, k, j, seed)
            if trained:
                done += trained
                print(f'seed {seed} | outer {k} | inner {j}: {trained} models in {(time.time()-t0)/60:.1f} min '
                      f'| {done}/{total} done | elapsed {(time.time()-t_start)/3600:.2f} h', flush=True)
print('all runs finished')

## 6. CV+ evaluation → summary on Drive

In [ ]:
def load_run(config, k, j, seed):
    z = np.load(f'{PRED_DIR}/{TAGS[config]}_outer{k}_inner{j}_s{seed}.npz')
    return {key: z[key] for key in z}

def complete(config, seed):
    return all(os.path.exists(f'{PRED_DIR}/{TAGS[config]}_outer{f["fold"]}_inner{j}_s{seed}.npz')
               for f in outer_folds for j in range(1, K_INNER + 1))

SCHEMES = {'none': 'single q', 'pred': 'predicted stage', 'true': 'true stage (oracle)'}
SUBSETS = ['healthy', 'early', 'nearfail', 'postFPT', 'overall']
summary = {}
for config in CONFIGS:
    seeds_ok = [s for s in SEEDS if complete(config, s)]
    per_seed = []
    for seed in seeds_ok:
        folds = []
        for of in outer_folds:
            k = of['fold']
            ev = evaluate_outer([load_run(config, k, j, seed) for j in range(1, K_INNER + 1)], ALPHA)
            y, yhat, st, bid = ev['point']
            grid = evaluate_grid(y, yhat, st, bid)
            row = {'rmse_' + s: grid[s]['per_bearing'] for s in SUBSETS}
            row['stage_acc'] = ev['stage_acc_test']; row['nf_recall'] = ev['nf_recall_test']
            for sc in SCHEMES:
                for s in SUBSETS:
                    for m in ('picp', 'mpiw', 'mpiw_raw'):
                        row[f'{sc}_{s}_{m}'] = ev[sc][s][m]
            folds.append(row)
        per_seed.append({key: float(np.nanmean([f[key] for f in folds])) for key in folds[0]})
    summary[config] = {'seeds': seeds_ok,
                       **{key: [float(np.mean([p[key] for p in per_seed])), float(np.std([p[key] for p in per_seed]))]
                          for key in (per_seed[0] if per_seed else {})}}

with open(f'{RESULTS_DIR}/cvplus_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=1)
print(f'saved {RESULTS_DIR}/cvplus_summary.json')

## 7. Tables

In [ ]:
def f(config, key, d=3):
    m, s = summary[config].get(key, [np.nan, np.nan]); return f'{m:.{d}f}±{s:.{d}f}'

lines = ['# CV+ ablation (alpha = %.2f), mean ± std over seeds\n' % ALPHA]
lines.append('| config | seeds | NF RMSE | overall RMSE | stage acc | NF recall |')
lines.append('|---|---|---|---|---|---|')
for c in CONFIGS:
    lines.append(f"| {c} | {len(summary[c]['seeds'])} | {f(c,'rmse_nearfail')} | {f(c,'rmse_overall')} "
                 f"| {f(c,'stage_acc')} | {f(c,'nf_recall')} |")
for s, lab in (('nearfail', 'near-failure'), ('postFPT', 'post-FPT'), ('overall', 'all')):
    lines.append(f'\n## Coverage / clipped width on {lab} windows\n')
    lines.append('| grouping | ' + ' | '.join(CONFIGS) + ' |')
    lines.append('|---|' + '---|' * len(CONFIGS))
    for sc, lab_sc in SCHEMES.items():
        lines.append(f'| {lab_sc} | ' + ' | '.join(
            f"{f(c, f'{sc}_{s}_picp')} / {summary[c].get(f'{sc}_{s}_mpiw', [np.nan])[0]:.2f}" for c in CONFIGS) + ' |')
report = '\n'.join(lines)
open(f'{RESULTS_DIR}/cvplus_report.md', 'w').write(report + '\n')
print(report)